In [1]:
import baostock as bs
import pandas as pd
from pathlib import Path

def fetch_daily_baostock(symbol: str, start: str, end: str):
    # 1. 转换日期格式 (baostock 需要 2020-01-01 这种格式)
    start_fmt = f"{start[:4]}-{start[4:6]}-{start[6:]}"
    end_fmt = f"{end[:4]}-{end[4:6]}-{end[6:]}"
    
    # 2. 转换股票代码 (6开头是上海sh，其他是深圳sz)
    if symbol.startswith('6'):
        bs_code = f"sh.{symbol}"
    else:
        bs_code = f"sz.{symbol}"
        
    # 3. 请求数据 (adjustflag="1" 表示后复权，跟之前的 hfq 一样)
    rs = bs.query_history_k_data_plus(bs_code,
        "date,code,open,high,low,close,volume,amount",
        start_date=start_fmt, end_date=end_fmt,
        frequency="d", adjustflag="1")
    
    # 4. 将数据整理成 DataFrame
    data_list = []
    while (rs.error_code == '0') & rs.next():
        data_list.append(rs.get_row_data())
    
    df = pd.DataFrame(data_list, columns=rs.fields)
    return df

if __name__ == "__main__":
    # 登录系统 (再次运行也不会冲突)
    lg = bs.login()
    
    out = Path("data/raw")
    out.mkdir(parents=True, exist_ok=True)
    
    for s in ["600519", "000001", "601318"]:
        print(f"正在从 Baostock 抓取 {s} 的数据...")
        try:
            df = fetch_daily_baostock(s, "20200101", "20261231")
            df.to_csv(out / f"{s}.csv", index=False)
            print(f"{s} 数据已保存！共 {len(df)} 条记录。")
        except Exception as e:
            print(f"抓取 {s} 失败: {e}")
            
    # 登出系统
    bs.logout()
    print("所有数据抓取完毕，已安全登出！")

login success!
正在从 Baostock 抓取 600519 的数据...
600519 数据已保存！共 1664 条记录。
正在从 Baostock 抓取 000001 的数据...
000001 数据已保存！共 1664 条记录。
正在从 Baostock 抓取 601318 的数据...
601318 数据已保存！共 1664 条记录。
logout success!
所有数据抓取完毕，已安全登出！
